# M6 · Optimisation — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. Here you implement gradient descent from scratch on the module's own two-well surface, sweep the learning rate until you can reproduce all three regimes on demand, watch minibatch noise at different batch sizes, and then run the same machinery on a real logistic regression to see the identical behaviours at scale.

Companion to the **Optimisation** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(6)

## 1 · The surface, and descent from scratch

The module widget's two-well landscape: $f(x,y) = (x^2-1)^2 + 0.3x + 0.5y^2$ — two basins of different depths with a saddle between them. Gradient descent is four lines of code.

In [ ]:
def f(p):
    x, y = p
    return (x**2 - 1)**2 + 0.3*x + 0.5*y**2

def grad(p):
    x, y = p
    return np.array([4*x*(x**2 - 1) + 0.3, y])

def descend(start, lr, steps=120):
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        p = p - lr * grad(p)
        path.append(p.copy())
        if not np.isfinite(f(p)) or abs(p[0]) > 10:
            break
    return np.array(path)

path = descend([1.6, 1.4], lr=0.05)
print(f"start loss {f(path[0]):.3f}  ->  final loss {f(path[-1]):.4f}  at  {path[-1].round(3)}")

## 2 · The learning-rate sweep: three regimes on one plot

Same start point, same surface, four learning rates. The trajectories are drawn on the contour map; the loss curves below them are the signatures to memorise — you will see them again on real training runs.

In [ ]:
xs = np.linspace(-2, 2, 200); ys = np.linspace(-2, 2, 200)
X, Y = np.meshgrid(xs, ys)
Z = (X**2 - 1)**2 + 0.3*X + 0.5*Y**2

lrs = [0.005, 0.05, 0.23, 0.55]
labels = ["too small: crawl", "well chosen", "oscillating", "diverged"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for k, (lr, lab) in enumerate(zip(lrs, labels)):
    p = descend([1.6, 1.4], lr=lr, steps=80)
    ax = axes[0, k]
    ax.contourf(X, Y, np.minimum(Z, 6), levels=18, cmap="Blues_r")
    ax.plot(p[:, 0], p[:, 1], ".-", color="tab:red", ms=3, lw=1)
    ax.set_title(f"lr={lr}: {lab}", fontsize=10)
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    losses = [f(q) for q in p]
    axes[1, k].plot(losses, color="tab:red")
    axes[1, k].set_yscale("log"); axes[1, k].set_xlabel("step")
axes[1, 0].set_ylabel("loss (log)")
plt.tight_layout(); plt.show()

The divergence threshold is not folklore — problem 2 derives it. Near the right-hand minimum the curvature along $x$ is about $f''(x^\*) \approx 8$, so descent should survive up to roughly $\eta = 2/8 = 0.25$ and explode beyond. Check it:

In [ ]:
for lr in [0.20, 0.24, 0.26, 0.30]:
    p = descend([0.96, 0.0], lr=lr, steps=60)   # start at the minimum's basin
    print(f"lr={lr}:  final loss {f(p[-1]):.3g}   ({'diverged' if f(p[-1]) > 10 else 'stable'})")

## 3 · Minibatch gradients are survey estimates

A fixed dataset defines a fixed average-loss surface; a minibatch estimates its gradient from a sample. Watch the estimate's noise shrink like $1/\sqrt{|B|}$ — the oldest law in survey statistics, running your optimiser.

In [ ]:
# A synthetic logistic-regression dataset: 2 features, N=4000
N = 4000
Xd = rng.standard_normal((N, 2))
true_w = np.array([2.0, -1.0])
yd = (1 / (1 + np.exp(-(Xd @ true_w))) > rng.random(N)).astype(float)

def logistic_grad(w, idx):
    z = Xd[idx] @ w
    pred = 1 / (1 + np.exp(-z))
    return Xd[idx].T @ (pred - yd[idx]) / len(idx)

w0 = np.zeros(2)
full = logistic_grad(w0, np.arange(N))
for B in [8, 32, 128, 512]:
    ests = np.array([logistic_grad(w0, rng.choice(N, B, replace=False)) for _ in range(400)])
    err = np.linalg.norm(ests - full, axis=1)
    print(f"batch {B:4d}:  mean gradient error {err.mean():.4f}   (1/sqrt(B) = {1/np.sqrt(B):.4f})")

## 4 · The same regimes, on a real model

Full-batch and minibatch gradient descent on the logistic regression. The signatures from section 2 — crawl, clean descent, oscillation, divergence — reappear unchanged: the surface is different, the handwriting is the same. The minibatch curve wobbles around the full-batch curve without departing from it.

In [ ]:
def logistic_loss(w):
    z = Xd @ w
    return np.mean(np.log(1 + np.exp(-z)) + (1 - yd) * z)

def train(lr, batch=None, steps=150):
    w = np.zeros(2); losses = []
    for t in range(steps):
        idx = np.arange(N) if batch is None else rng.choice(N, batch, replace=False)
        w = w - lr * logistic_grad(w, idx)
        losses.append(logistic_loss(w))
        if not np.isfinite(losses[-1]) or losses[-1] > 50: break
    return losses

fig, ax = plt.subplots(figsize=(8, 4.5))
for lr, style, lab in [(0.05, "-", "lr=0.05 crawl"), (1.0, "-", "lr=1.0 good"), (14.0, "-", "lr=14 oscillate"), (30.0, "--", "lr=30 diverge")]:
    ax.plot(train(lr), style, label=lab)
ax.plot(train(1.0, batch=32), ":", label="lr=1.0, batch=32 (noisy)")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("loss (log)")
ax.legend(fontsize=9); ax.set_title("Logistic regression: the same signatures at scale")
plt.tight_layout(); plt.show()

## 5 · Momentum on a ravine

The widget's ravine, $f = \tfrac{1}{2}(x^2 + 15y^2)$: curvature 1 along the floor, 15 across it. The same learning rate zigzags without momentum and glides with it — agreement compounds, alternation cancels.

In [ ]:
def ravine_grad(p): return np.array([p[0], 15*p[1]])

def descend_ravine(lr, beta=0.0, steps=60):
    p = np.array([-1.8, 0.9]); v = np.zeros(2); path = [p.copy()]
    for _ in range(steps):
        v = beta*v - lr*ravine_grad(p)
        p = p + v
        path.append(p.copy())
    return np.array(path)

Zr = 0.5*(X**2 + 15*Y**2)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, beta, title in [(axes[0], 0.0, "lr=0.12, no momentum"), (axes[1], 0.85, "lr=0.12, momentum β=0.85")]:
    p = descend_ravine(0.12, beta)
    ax.contourf(X, Y, np.minimum(Zr, 12), levels=18, cmap="Blues_r")
    ax.plot(p[:, 0], p[:, 1], ".-", color="tab:red", ms=3, lw=1)
    ax.set_title(title, fontsize=10); ax.set_xlim(-2, 2); ax.set_ylim(-1.2, 1.2)
plt.tight_layout(); plt.show()

## 6 · A five-line learning-rate finder

The practitioner's move: sweep the learning rate geometrically, run a few steps at each, and plot the final loss. The usable band sits between the crawl and the cliff — pick something inside it, conventionally a bit below the minimum of the curve.

In [ ]:
rates = np.logspace(-3, 1.6, 30)
finals = [train(lr, batch=64, steps=40)[-1] for lr in rates]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rates, finals, ".-")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("learning rate"); ax.set_ylabel("loss after 40 steps")
ax.set_title("The usable band: between the crawl and the cliff")
plt.tight_layout(); plt.show()

Everything in this notebook — the regimes, the noise, the ravine, the sweep — is what a real training run does, minus only scale. When the deep-learning track (D5) shows you pathological loss curves from actual networks, you have already produced every one of them by hand here.

---

**Next:** M7 · Probability for Language — from inference over parameters to generation over sequences.